In [11]:
import pandas as pd
from datetime import date
import psycopg2

df = []

with open(r'c:\Users\H59045\Downloads\llmonliq_202608\llmonliq_202608.txt', 'r') as f:
    lines = [line for line in f if line.startswith('LR')]

from io import StringIO

data= ''.join(lines)
df=pd.read_csv(StringIO(data), sep='|', header=None)

print(df.head())

   0          1       2  3   4         5           6     7       8  9   10  \
0  LR  36202CRD7  002284  M  SF  19960901  1023444403  3871  202608  F   1   
1  LR  36202CRD7  002284  M  SF  19960901  1023451366  4042  202608  F   1   
2  LR  36202CRE5  002285  M  SF  19960901  1023505445  3162  202608  V   1   
3  LR  36202CRW5  002301  M  SF  19961001  1023373921  4094  202608  F   1   
4  LR  36202CRW5  002301  M  SF  19961001  1023377348  3162  202608  F   1   

         11      12      13  14  
0  119000.0  392.37    0.00   1  
1   48000.0  626.09    0.00   1  
2   68000.0  210.09    0.00   1  
3   65000.0  884.47  411.97   1  
4   64000.0  141.35    0.00   1  


In [12]:
headers = ['record_type', 'cusip_number', 'pool_id','pool_indicator', 'pool_type', 'pool_issue_date', 'disclosure_sequence_number', 'issuer_id', 'report_period', 'agency_type', 'loan_purpose', 'original_principal_balance', 'unpaid_principal_balance', 'liquidation_balance', 'removal_reason']

df.columns = headers

print(df.head())


  record_type cusip_number pool_id pool_indicator pool_type  pool_issue_date  \
0          LR    36202CRD7  002284              M        SF         19960901   
1          LR    36202CRD7  002284              M        SF         19960901   
2          LR    36202CRE5  002285              M        SF         19960901   
3          LR    36202CRW5  002301              M        SF         19961001   
4          LR    36202CRW5  002301              M        SF         19961001   

   disclosure_sequence_number  issuer_id  report_period agency_type  \
0                  1023444403       3871         202608           F   
1                  1023451366       4042         202608           F   
2                  1023505445       3162         202608           V   
3                  1023373921       4094         202608           F   
4                  1023377348       3162         202608           F   

   loan_purpose  original_principal_balance  unpaid_principal_balance  \
0             1    

In [13]:
import psycopg2.extras


conn = psycopg2.connect(dbname='postgres', user='postgres', password=input(), host='localhost') 
cursor = conn.cursor()

# Insert dataframe
for index, row in df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO temp_liquidations VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """, tuple(row))
    except Exception as e:
        print(f"Error on row {index}: {e}")
        print(f"Row values: {tuple(row)}")
        print(f"Row length: {len(tuple(row))}")
        break

conn.commit()